# QVerse — Introduction to Quantum Computing & Programming
        ## Week 14: Noise, Transpilation, and Real Hardware

        **Level:** Beginner  
        **Recommended study time:** 2–4 hours  
        **Prerequisites:** Weeks 1–13

        ### Learning objectives
        - Explain why ideal and hardware circuits differ.
- Transpile circuits for a restricted target.
- Compare depth and operation counts under optimization.
- Simulate a simple noisy Bell experiment when Qiskit Aer is available.

        ---
        **How to use this notebook**

        1. Read the short theory sections.
        2. Make a prediction before running each guided experiment.
        3. Run and modify the code.
        4. Complete every **TODO** exercise.
        5. Finish the reflection section in your own words.

        The goal is not to memorize syntax. The goal is to connect **quantum idea → circuit → result → explanation**.

In [ ]:
# Run this only if your environment does not have the required packages.
# In a terminal, the preferred setup is:
# python -m pip install "qiskit[visualization]>=2.5" matplotlib numpy

# In a fresh Colab notebook you can instead uncomment:
# %pip install "qiskit[visualization]>=2.5" matplotlib numpy -q

## 1. Logical circuit versus physical implementation

A circuit diagram is a logical description. Real processors impose:
- a limited native gate set;
- restricted qubit connectivity;
- gate and readout errors;
- finite coherence times.

**Transpilation** rewrites a circuit into operations compatible with a target while trying to preserve the computation and optimize chosen metrics.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap
from qiskit.quantum_info import Statevector

## 2. A routing example without needing cloud access

In [ ]:
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 2)   # non-neighbor interaction for a linear 0-1-2 topology
qc.cx(2, 0)

coupling = CouplingMap([[0,1], [1,0], [1,2], [2,1]])

print("Original depth:", qc.depth())
print("Original ops:", qc.count_ops())

tqc = transpile(
    qc,
    basis_gates=["rz", "sx", "x", "cx"],
    coupling_map=coupling,
    optimization_level=1,
    seed_transpiler=11,
)

print("Transpiled depth:", tqc.depth())
print("Transpiled ops:", tqc.count_ops())
tqc.draw("mpl")

## 3. Optimization-level comparison

In [ ]:
for level in range(4):
    tqc = transpile(
        qc,
        basis_gates=["rz", "sx", "x", "cx"],
        coupling_map=coupling,
        optimization_level=level,
        seed_transpiler=11,
    )
    print(level, "depth =", tqc.depth(), "ops =", dict(tqc.count_ops()))

## 4. Optional: noisy simulation with Qiskit Aer

In [ ]:
# If needed:
# %pip install qiskit-aer -q

try:
    from qiskit_aer import AerSimulator
    from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
    AER_AVAILABLE = True
except ImportError:
    AER_AVAILABLE = False
    print("qiskit-aer is not installed. The rest of this notebook still works.")

In [ ]:
if AER_AVAILABLE:
    from qiskit.visualization import plot_histogram

    bell = QuantumCircuit(2)
    bell.h(0)
    bell.cx(0,1)
    bell.measure_all()

    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 1), ["h", "x", "sx", "rz"])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(0.03, 2), ["cx"])
    readout = ReadoutError([[0.97, 0.03], [0.04, 0.96]])
    noise_model.add_all_qubit_readout_error(readout)

    ideal_backend = AerSimulator()
    noisy_backend = AerSimulator(noise_model=noise_model)

    ideal_counts = ideal_backend.run(transpile(bell, ideal_backend), shots=4000).result().get_counts()
    noisy_counts = noisy_backend.run(transpile(bell, noisy_backend), shots=4000).result().get_counts()

    print("Ideal:", ideal_counts)
    print("Noisy:", noisy_counts)
    plot_histogram([ideal_counts, noisy_counts], legend=["ideal", "noisy"])

## 5. Real IBM hardware — optional extension

Running on IBM Quantum hardware requires a current IBM Quantum account and `qiskit-ibm-runtime`. The exact service/account setup can change, so use the current IBM Quantum documentation when mentors enable this extension.

The conceptual workflow is:
1. choose an accessible backend;
2. transpile to that backend's ISA;
3. execute with a Runtime primitive;
4. compare hardware data with ideal expectations;
5. document backend, shots, compilation settings, and date.

## Core exercises
1. Measure circuit depth before and after transpilation for the linear coupling map.
2. Explain why a logical interaction between non-neighboring qubits may require routing.
3. Compare optimization levels 0–3 and record depth and two-qubit-gate counts.
4. If Aer is available, compare ideal and noisy Bell counts and quantify the fraction of undesired outcomes.

In [ ]:
# TODO: Write your solutions here.
# Add extra code cells when useful.

## Optional stretch challenge
Create two logically equivalent circuits with noticeably different depths or two-qubit-gate counts. Under the same noise model, compare their output quality.

In [ ]:
# OPTIONAL TODO: Attempt the stretch challenge here.

## Weekly reflection
- What problem does transpilation solve?
- Why can two mathematically equivalent circuits perform differently on hardware?
- Why is 'error mitigation' not the same as perfect error correction?

## Submission checklist
- [ ] I made at least one prediction before executing a circuit.
- [ ] All guided examples run.
- [ ] I completed the core exercises.
- [ ] I explained the important output rather than only displaying it.
- [ ] My notebook is readable from top to bottom.